# Gemini Enterprise + Model Armor — Dashboard Setup (reproducible)

이 노트북은 `gemini_ent_analytics`(Log Analytics linked dataset) → `gemini_ent_dashboard`(뷰) →
BigQuery ML 원격 모델 → Looker Studio 대시보드까지 전 과정을 재현합니다.

- 각 셀은 **멱등(idempotent)** 하게 작성되어 있습니다: 이미 리소스가 있으면 건너뛰거나 `CREATE OR REPLACE`로 대체합니다.
- Terraform과 동일한 리소스를 다루므로, 운영 환경에서는 **Terraform 쪽을 단일 진실 공급원(source of truth)** 으로 쓰고
  이 노트북은 탐색/디버깅/1회성 실행용으로 사용하는 것을 권장합니다.
- 실제 GCP 리소스를 만들고 싶지 않다면 각 코드 셀을 실행하지 말고 읽기만 하세요.

**주의 (forward-only)**: Log Analytics는 활성화 시점 이후의 로그만 인덱싱합니다. 과거 로그는 백필되지 않습니다.

**비용 주의**: 5단계(콘텐츠 분류)와 6-B단계(콘텐츠 분류 예약 쿼리)는 행당 Gemini 호출 1회가 발생합니다.
기본값은 둘 다 비활성화입니다. 원격 모델(`gemini_flash`)은 비용 절감을 위해 `gemini-2.5-flash-lite`
엔드포인트를 사용합니다 (이 프로젝트에서 `gemini-2.0-*` 계열 엔드포인트는 not found로 실패함이 실측 확인됨).


## 0) 변수 설정

프로젝트/데이터셋/커넥션/모델 이름을 한 곳에서 관리합니다. Terraform의 `variables.tf`와 값이 동일합니다.


In [ ]:
# (Colab 전용) 필요한 라이브러리가 없다면 설치
try:
    import google.cloud.bigquery  # noqa: F401
except ImportError:
    %pip install --quiet google-cloud-bigquery


In [ ]:
import os
import subprocess

PROJECT_ID = "YOUR_PROJECT_ID"
BQ_LOCATION = "US"

DASHBOARD_DATASET = "gemini_ent_dashboard"
ANALYTICS_DATASET = "gemini_ent_analytics"          # Log Analytics linked dataset (table: _AllLogs)

LOG_BUCKET_NAME = "_Default"
LOG_BUCKET_LOCATION = "global"

CONNECTION_ID = "gemini_conn"                        # us.gemini_conn (CLOUD_RESOURCE)
MODEL_NAME = "gemini_flash"
# Flash-Lite로 설정: Flash 대비 호출당 비용이 저렴함 (콘텐츠 분류에서 행당 1회 호출되므로 중요).
# 실측 확인(이 프로젝트, YOUR_PROJECT_ID): gemini-2.0-flash, gemini-2.0-flash-lite,
# gemini-2.0-flash-001 은 모두 "not found"로 실패한다. gemini-2.5-flash-lite / gemini-2.5-flash 만 작동 확인됨.
# 2.0 계열로 되돌리지 말 것 (재검증 없이는).
MODEL_ENDPOINT = "gemini-2.5-flash-lite"

# 비용 유발 단계(5번, 콘텐츠 분류)를 실제로 실행할지 여부. 기본은 False.
RUN_CONTENT_CLASSIFICATION = False

# 6-B 단계(예약 쿼리로 콘텐츠 분류 자동화)를 실제로 생성할지 여부. 기본은 False.
# 주의: 이미 실환경에 동일 목적의 예약 쿼리가 수동으로 생성되어 있음
# (transferConfig id YOUR_TRANSFER_CONFIG_ID, location us,
#  Compute Engine 기본 서비스 계정으로 실행, schedule "every day 18:00").
# True로 켜기 전에 아래 6-B 셀의 중복 생성 경고를 반드시 읽으세요.
ENABLE_SCHEDULED_CLASSIFICATION = False

# 예약 쿼리 스케줄 (BigQuery Data Transfer Service 문법). 실환경 config와 동일하게 맞춤.
# 주의: BigQuery 예약 쿼리는 항상 UTC 기준입니다 (타임존 파라미터 없음).
# "every day 18:00" = 18:00 UTC = 03:00 KST (다음날). Terraform var.scheduled_query_schedule과 동일.
SCHEDULED_QUERY_SCHEDULE = "every day 18:00"

# 이 노트북 파일 기준 repo 루트 추정 (sql/ 디렉토리를 찾을 수 있는 경로)
CANDIDATE_ROOTS = [os.getcwd(), os.path.join(os.getcwd(), ".."), "/content/dev-geminienterprise-dashboard"]
REPO_ROOT = None
for c in CANDIDATE_ROOTS:
    if os.path.isfile(os.path.join(c, "sql", "01_create_views.sql")):
        REPO_ROOT = os.path.abspath(c)
        break

if REPO_ROOT is None:
    print("sql/01_create_views.sql 을 찾지 못했습니다. REPO_ROOT를 직접 지정하거나,")
    print("리포지토리를 클론한 뒤 이 셀을 다시 실행하세요, 예:")
    print('  !git clone <repo-url> /content/dev-geminienterprise-dashboard')
else:
    print("REPO_ROOT =", REPO_ROOT)

SQL_CREATE_VIEWS = os.path.join(REPO_ROOT, "sql", "01_create_views.sql") if REPO_ROOT else None
SQL_CONTENT_CLASSIFICATION = os.path.join(REPO_ROOT, "sql", "02_content_classification.sql") if REPO_ROOT else None

# sql/01_create_views.sql and sql/02_content_classification.sql are plain
# .sql files meant to also be runnable standalone (pasted into the BQ
# console, or `bq query < sql/01_create_views.sql`) against the project they
# were authored against, so they intentionally keep that project id
# hardcoded rather than a template placeholder:
_ORIGINAL_PROJECT_ID = "YOUR_PROJECT_ID"

def load_sql(path):
    """Read a sql/ file and rewrite its hardcoded project id to PROJECT_ID,
    so this notebook also works unmodified against a different project.
    No-op (byte-for-byte identical) when PROJECT_ID == _ORIGINAL_PROJECT_ID.
    """
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    return text.replace(_ORIGINAL_PROJECT_ID, PROJECT_ID)

print("PROJECT_ID:", PROJECT_ID)
print("BQ_LOCATION:", BQ_LOCATION)


In [ ]:
# 인증 (Colab 환경이면 사용자 계정으로 로그인, 이미 gcloud ADC가 있으면 스킵)
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab 사용자 인증 완료")
else:
    print("Colab이 아님 — 로컬/Compute Engine ADC 또는 `gcloud auth application-default login` 자격 증명을 사용합니다.")

from google.cloud import bigquery
bq_client = bigquery.Client(project=PROJECT_ID)
print("BigQuery client ready for project:", bq_client.project)


## 1) Log Analytics 활성화 + Linked Dataset 생성

- `_Default` 로그 버킷에서 Log Analytics를 활성화합니다 (한 번 켜면 되돌릴 수 없음 — 멱등하게 이미 켜져 있으면 스킵).
- 그 다음 `gemini_ent_analytics` linked dataset을 만듭니다 (테이블 `_AllLogs` 생성됨).
- **forward-only 주의**: 활성화 이후 로그만 `_AllLogs`에 인덱싱됩니다. 과거 로그는 백필되지 않습니다.


In [ ]:
import json as _json

def _sh(cmd):
    """Run a shell command, return (returncode, stdout+stderr)."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr)

# --- 1a. Log Analytics 활성화 (멱등: 이미 활성화면 스킵) ---
rc, out = _sh(
    f"gcloud logging buckets describe {LOG_BUCKET_NAME} "
    f"--location={LOG_BUCKET_LOCATION} --project={PROJECT_ID} "
    f"--format='value(analyticsEnabled)'"
)
already_enabled = out.strip() == "True"
print("Log Analytics already enabled:", already_enabled)

if not already_enabled:
    rc, out = _sh(
        f"gcloud logging buckets update {LOG_BUCKET_NAME} "
        f"--project={PROJECT_ID} --location={LOG_BUCKET_LOCATION} --enable-analytics"
    )
    print(out)
    if rc != 0:
        raise RuntimeError("Log Analytics enable 실패 — 로그를 확인하세요.")
else:
    print("스킵: 이미 활성화되어 있습니다.")


In [ ]:
# --- 1b. Linked dataset 생성 (멱등: 이미 있으면 스킵) ---
rc, out = _sh(
    f"gcloud logging links list --bucket={LOG_BUCKET_NAME} "
    f"--location={LOG_BUCKET_LOCATION} --project={PROJECT_ID} "
    f"--format='value(NAME)'"
)
existing_links = [l.strip() for l in out.splitlines() if l.strip()]
print("Existing links:", existing_links)

if ANALYTICS_DATASET not in existing_links:
    rc, out = _sh(
        f"gcloud logging links create {ANALYTICS_DATASET} "
        f"--bucket={LOG_BUCKET_NAME} --location={LOG_BUCKET_LOCATION} "
        f"--project={PROJECT_ID} "
        f'--description="Log Analytics linked dataset for Gemini Enterprise + Model Armor logs"'
    )
    print(out)
    if rc != 0:
        raise RuntimeError("Linked dataset 생성 실패 — 로그를 확인하세요.")
else:
    print(f"스킵: linked dataset '{ANALYTICS_DATASET}' 이 이미 존재합니다.")

print()
print("⚠️  forward-only: 이 시점 이후의 로그만 이제부터 _AllLogs에 쌓입니다. 과거 로그는 백필되지 않습니다.")


## 2) 대시보드 데이터셋 생성

뷰(`v_*`), 콘텐츠 분류 테이블, 원격 모델이 들어갈 `gemini_ent_dashboard` 데이터셋을 생성합니다 (US 멀티 리전).


In [ ]:
from google.cloud import bigquery as _bq
from google.api_core.exceptions import Conflict, NotFound

dataset_ref = f"{PROJECT_ID}.{DASHBOARD_DATASET}"
try:
    bq_client.get_dataset(dataset_ref)
    print(f"스킵: 데이터셋 {dataset_ref} 이미 존재합니다.")
except NotFound:
    ds = _bq.Dataset(dataset_ref)
    ds.location = BQ_LOCATION
    ds.description = (
        "Gemini Enterprise + Model Armor dashboard: views (v_*), "
        "content-classification table (t_content_topics), gemini_flash remote model."
    )
    ds.labels = {"app": "gemini-enterprise-dashboard", "layer": "presentation"}
    bq_client.create_dataset(ds)
    print(f"생성됨: {dataset_ref} (location={BQ_LOCATION})")


## 3) 뷰 생성 (`sql/01_create_views.sql` 실행)

`bq show --view`로 추출해 통합한 19개 뷰 정의를 한 번에 실행합니다. `CREATE OR REPLACE VIEW`라서 재실행해도 안전합니다(멱등).


In [ ]:
assert SQL_CREATE_VIEWS and os.path.isfile(SQL_CREATE_VIEWS), (
    "sql/01_create_views.sql 을 찾을 수 없습니다. REPO_ROOT 설정을 확인하세요."
)

views_script = load_sql(SQL_CREATE_VIEWS)  # rewrites hardcoded project id to PROJECT_ID

print(f"{len(views_script):,} bytes 스크립트 실행 중 (19개 CREATE OR REPLACE VIEW)...")
job = bq_client.query(views_script, location=BQ_LOCATION)
job.result()  # 완료까지 대기
print("완료. job id:", job.job_id)


In [ ]:
# 검증: 뷰 개수 확인
views = [t.table_id for t in bq_client.list_tables(f"{PROJECT_ID}.{DASHBOARD_DATASET}") if t.table_type == "VIEW"]
print(f"{len(views)}개 뷰 확인됨:")
for v in sorted(views):
    print(" -", v)


## 4) BQ 커넥션 + IAM + 원격 모델

1. `us.gemini_conn` (CLOUD_RESOURCE) 커넥션 생성 (멱등: 있으면 스킵)
2. 커넥션의 서비스 계정에 `roles/aiplatform.user` 부여 (멱등: `gcloud` add-iam-policy-binding은 이미 있으면 no-op)
3. `gemini_ent_dashboard.gemini_flash` 원격 모델 생성 (`CREATE OR REPLACE MODEL`이라 재실행 안전)


In [ ]:
# --- 4a. 커넥션 생성 (멱등) ---
rc, out = _sh(
    f"bq show --connection --format=json {PROJECT_ID}.{BQ_LOCATION}.{CONNECTION_ID}"
)
if rc == 0:
    conn_info = _json.loads(out)
    conn_sa = conn_info["cloudResource"]["serviceAccountId"]
    print(f"스킵: 커넥션 {CONNECTION_ID} 이미 존재합니다. SA={conn_sa}")
else:
    rc, out = _sh(
        f"bq mk --connection --connection_type=CLOUD_RESOURCE "
        f"--project_id={PROJECT_ID} --location={BQ_LOCATION} {CONNECTION_ID}"
    )
    print(out)
    if rc != 0:
        raise RuntimeError("커넥션 생성 실패")
    rc, out = _sh(f"bq show --connection --format=json {PROJECT_ID}.{BQ_LOCATION}.{CONNECTION_ID}")
    conn_sa = _json.loads(out)["cloudResource"]["serviceAccountId"]
    print("생성된 커넥션 SA:", conn_sa)


In [ ]:
# --- 4b. IAM: 커넥션 SA에 roles/aiplatform.user 부여 (멱등) ---
rc, out = _sh(
    f"gcloud projects add-iam-policy-binding {PROJECT_ID} "
    f'--member="serviceAccount:{conn_sa}" '
    f'--role="roles/aiplatform.user" --condition=None --format=none'
)
print(out if out.strip() else "IAM 바인딩 적용 완료 (이미 있었다면 no-op)")
if rc != 0:
    raise RuntimeError("IAM 바인딩 실패 — roles/resourcemanager.projectIamAdmin 권한을 확인하세요.")


In [ ]:
# --- 4c. 원격 모델 생성 (CREATE OR REPLACE라 멱등) ---
create_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DASHBOARD_DATASET}.{MODEL_NAME}`
REMOTE WITH CONNECTION `{PROJECT_ID}.{BQ_LOCATION}.{CONNECTION_ID}`
OPTIONS (ENDPOINT = '{MODEL_ENDPOINT}');
"""
print(create_model_sql)

job = bq_client.query(create_model_sql, location=BQ_LOCATION)
job.result()
print("모델 생성/갱신 완료:", f"{PROJECT_ID}.{DASHBOARD_DATASET}.{MODEL_NAME}")


## 5) (옵트인) ② 콘텐츠 인텔리전스 — 토픽/의도/감성 분류

⚠️ **비용 경고**: 이 단계는 아직 분류되지 않은 프롬프트 각 행마다 Gemini 호출 1회가 발생합니다
(`sql/02_content_classification.sql`은 증분 실행이라 이미 분류된 timestamp는 건너뜁니다. 그래도
신규 로그가 쌓일 때마다 실행 비용이 발생합니다).

기본적으로 `RUN_CONTENT_CLASSIFICATION = False`로 설정되어 있어 이 셀은 아무 것도 하지 않습니다.
실행하려면 위 0단계 변수 셀에서 `RUN_CONTENT_CLASSIFICATION = True`로 바꾸고 이 셀부터 다시 실행하세요.

ℹ️ **참고**: `t_content_topics`는 `v_*` 뷰들과 달리 **자동으로 최신화되지 않는 append-only 테이블**입니다.
이 셀(또는 아래 6-B 단계의 예약 쿼리)을 주기적으로 실행하지 않으면 신규 로그가 분류되지 않은 채 쌓입니다.


In [ ]:
if not RUN_CONTENT_CLASSIFICATION:
    print("스킵됨: RUN_CONTENT_CLASSIFICATION=False (비용 유발 단계). 실행하려면 0단계에서 True로 변경하세요.")
else:
    assert SQL_CONTENT_CLASSIFICATION and os.path.isfile(SQL_CONTENT_CLASSIFICATION), (
        "sql/02_content_classification.sql 을 찾을 수 없습니다."
    )
    classification_script = load_sql(SQL_CONTENT_CLASSIFICATION)  # rewrites hardcoded project id to PROJECT_ID

    print("⚠️  Gemini 호출이 실제로 발생합니다 (행당 1회). 계속 진행합니다...")
    job = bq_client.query(classification_script, location=BQ_LOCATION)
    job.result()
    print("완료. job id:", job.job_id)

    # 결과 요약
    q = f"""
    SELECT day, COUNT(*) AS classified_rows
    FROM `{PROJECT_ID}.{DASHBOARD_DATASET}.t_content_topics`
    GROUP BY day ORDER BY day DESC LIMIT 10
    """
    for row in bq_client.query(q, location=BQ_LOCATION).result():
        print(dict(row))


## 6) Looker Studio 대시보드 생성 URL

아래 셀은 `gemini_ent_dashboard`의 모든 뷰를 데이터 소스로 등록한 Looker Studio "새 보고서 만들기" 딥링크를
프로그램적으로 만듭니다. 이미 만들어둔 URL은 `looker_studio_create_url.txt`에도 있습니다. 출력된 URL을
브라우저에서 열면 각 뷰가 데이터 소스로 미리 채워진 상태로 보고서 편집 화면이 열립니다. 대시보드 레이아웃
구성 가이드는 `looker_studio_setup.md`를 참고하세요.


In [ ]:
import urllib.parse

VIEWS = [
    "v_daily_queries", "v_daily_queries_by_method", "v_daily_agent_calls",
    "v_queries_per_user", "v_daily_active_users", "v_daily_failure_rate",
    "v_streamassist_state", "v_model_armor_block", "v_model_armor_threats",
    "v_model_armor_threats_long", "v_hourly_heatmap", "v_agent_usage_daily",
    "v_response_latency_daily", "v_grounding_top_sources",
    "v_grounding_coverage_daily", "v_user_activity_detail",
    "v_model_armor_by_client", "v_user_agent_trace",
    "v_model_armor_verdict_daily",
]

report_name = "Gemini Enterprise 운영·보안 대시보드"
params = [
    ("c.mode", "edit"),
    ("c.reportName", report_name),
]
for v in VIEWS:
    params += [
        (f"ds.{v}.connector", "bigQuery"),
        (f"ds.{v}.type", "TABLE"),
        (f"ds.{v}.projectId", PROJECT_ID),
        (f"ds.{v}.billingProjectId", PROJECT_ID),
        (f"ds.{v}.datasetId", DASHBOARD_DATASET),
        (f"ds.{v}.tableId", v),
    ]

url = "https://lookerstudio.google.com/reporting/create?" + urllib.parse.urlencode(params)
print(url)


## 6-B) (옵트인) 콘텐츠 분류 예약 쿼리 (daily scheduled query)

`t_content_topics`는 append-only 테이블이라 `sql/02_content_classification.sql`을 누군가
주기적으로 실행해줘야 최신 상태를 유지합니다. 5단계처럼 노트북/Terraform apply 시점에 1회
실행하는 대신, BigQuery Data Transfer Service에 **`SCHEDULED_QUERY_SCHEDULE`(기본
`"every day 18:00"`) 스케줄로 자동 실행되는 예약 쿼리**로 등록해두는 옵션입니다.

⚠️ **BigQuery 예약 쿼리는 항상 UTC 기준입니다** (타임존 파라미터 없음). 기본값 `18:00`은
**18:00 UTC = 03:00 KST(다음날)** 을 의미합니다. 다른 시각을 원하면 0단계에서
`SCHEDULED_QUERY_SCHEDULE`을 UTC 기준으로 직접 환산해 바꾸세요.

⚠️ **이미 실환경에 동일 목적의 예약 쿼리가 수동으로 생성되어 있습니다**:
`projects/YOUR_PROJECT_NUMBER/locations/us/transferConfigs/YOUR_TRANSFER_CONFIG_ID`
(location `us`, Compute Engine 기본 서비스 계정으로 실행, schedule `every day 18:00` UTC —
원래 `every 24 hours`였다가 이 고정 시각 스케줄로 이미 갱신됨).
아래 셀을 그대로 실행하면 **중복** 예약 쿼리가 생성되어 분류 스크립트가 이중 실행/이중 과금됩니다.
`ENABLE_SCHEDULED_CLASSIFICATION = True`로 바꾸기 전에 기존 config를 재사용할지, 대체할지 먼저
`bq ls --transfer_config`로 확인하세요.

선행 조건: 프로젝트에 `bigquerydatatransfer.googleapis.com` API가 활성화되어 있어야 합니다
(`gcloud services enable bigquerydatatransfer.googleapis.com`).


In [ ]:
if not ENABLE_SCHEDULED_CLASSIFICATION:
    print("스킵됨: ENABLE_SCHEDULED_CLASSIFICATION=False. 켜기 전에 위 마크다운의 중복 생성 경고를 반드시 확인하세요.")
else:
    # 기존 config 중복 여부 먼저 확인
    rc, out = _sh(
        f"bq ls --transfer_config --transfer_location=us --project_id={PROJECT_ID} --format=json"
    )
    existing_configs = _json.loads(out) if rc == 0 and out.strip() else []
    dup = [c for c in existing_configs if c.get("displayName") == "Gemini Ent - Daily Content Classification"]
    if dup:
        print("⚠️  이미 동일한 이름의 예약 쿼리가 존재합니다 — 생성을 건너뜁니다:")
        for c in dup:
            print(" -", c.get("name"), "schedule:", c.get("schedule"))
    else:
        assert SQL_CONTENT_CLASSIFICATION and os.path.isfile(SQL_CONTENT_CLASSIFICATION), (
            "sql/02_content_classification.sql 을 찾을 수 없습니다."
        )
        classification_script = load_sql(SQL_CONTENT_CLASSIFICATION)  # rewrites hardcoded project id to PROJECT_ID

        params_json = _json.dumps({"query": classification_script})
        # 파라미터를 임시 파일에 써서 bq mk --params=@file 로 전달 (긴 SQL을 쉘 인자로 직접 넘기지 않기 위함)
        params_path = "/tmp/scheduled_query_params.json"
        with open(params_path, "w", encoding="utf-8") as f:
            f.write(params_json)

        rc, out = _sh(
            f"bq mk --transfer_config "
            f"--project_id={PROJECT_ID} "
            f"--target_dataset={DASHBOARD_DATASET} "
            f"--display_name=\"Gemini Ent - Daily Content Classification\" "
            f"--data_source=scheduled_query "
            f'--schedule="{SCHEDULED_QUERY_SCHEDULE}" '
            f"--params=@{params_path}"
        )
        print(out)
        if rc != 0:
            raise RuntimeError("예약 쿼리 생성 실패 — bigquerydatatransfer API 활성화 여부와 권한을 확인하세요.")
        print("생성 완료. `bq ls --transfer_config --transfer_location=us`로 확인하세요.")
